# Compliance Data — Search without GPU

Public-only corpus: 38 frameworks → 1519 chunks (512/50). Works in Colab, no `torch`, no `pixi`.

**Data:** `data/silver/compliance_chunks.parquet` (1.6M) from [Releases](https://github.com/dataengineergaurav/open-regulatory-corpus/releases) or local `data/silver/`

This notebook shows: load → filter by framework → keyword search (TF-IDF, no embeddings) → cite source.

> Not legal advice. Check `source` URLs for authoritative text.

In [ ]:
import pandas as pd
from pathlib import Path

# local or download from Release
parquet = Path("../data/silver/compliance_chunks.parquet")
if not parquet.exists():
    # fallback: download latest Release (no auth)
    url = "https://github.com/dataengineergaurav/open-regulatory-corpus/releases/latest/download/compliance-silver-2026-09-01.parquet"
    # try generic latest — if 404, use local path hint
    print(f"{parquet} not found — download from Releases: {url}")
    # uncomment to fetch:
    # import urllib.request; urllib.request.urlretrieve(url, "compliance_chunks.parquet"); parquet = Path("compliance_chunks.parquet")
else:
    df = pd.read_parquet(parquet)
    print(f"rows={len(df)} cols={list(df.columns)}")
    df.head(2)

In [ ]:
# distribution — what's actually in Silver
df["framework_id"].value_counts().head(15)

In [ ]:
# filter by framework — e.g. FDA AI-enabled medical devices
fda = df[df.framework_id == "FDA-AI-MD"]
print(f"FDA-AI-MD: {len(fda)} chunks")
print(fda.iloc[0]["text"][:800])
print("source:", fda.iloc[0]["source"])

In [ ]:
# keyword search without embeddings — TF-IDF (sklearn) or simple contains
# ponytail: sklearn if available, else pure pandas
query = "incident response logging"
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(df["text"].tolist())
    q = vec.transform([query])
    scores = cosine_similarity(q, X).flatten()
    top = scores.argsort()[::-1][:5]
    for i in top:
        print(f"{scores[i]:.3f} {df.iloc[i]['framework_id']:15} {df.iloc[i]['chunk_id']}")
        print(df.iloc[i]["text"][:400].replace("\n"," ")[:400])
        print("—", df.iloc[i]["source"])
        print()
except ImportError:
    # fallback: substring
    hits = df[df.text.str.contains(query, case=False, na=False)].head(5)
    for _, r in hits.iterrows():
        print(r["framework_id"], r["chunk_id"])
        print(r["text"][:400])
        print("—", r["source"], "\n")

In [ ]:
# cross-framework example: NGO wants all privacy + AI governance chunks
privacy = ["CCPA-CPRA","FERPA","COPPA","AU-PRIVACY-AI","UK-DP-AI","DOJ-DSP"]
ai_gov = ["NIST-AI-RMF","NIST-AI-600-1","OMB-M25-21","OWASP-LLM-2026"]
subset = df[df.framework_id.isin(privacy + ai_gov)]
print(f"privacy+ai subset: {len(subset)} chunks")
subset["framework_id"].value_counts()

## Next
- Replace TF-IDF with embeddings later: `sentence-transformers/all-MiniLM-L6-v2` + FAISS (Gold phase).
- Cite `source` + `sha256` per chunk — every answer should link to `framework_id` provenance in `data/sources.json`.
- WAF gaps: `HIPAA`, `CMMC`, `HHS-PART2`, `IL-AIVIA`, `EU-AI-ACT`, `GDPR`, `BR-LGPD` currently 0 chunks (see `docs/DATA_PIPELINE.md`).